# Iris Classifier Training

Simple classification example for Piper e2e testing.
Outputs `PIPER_METRIC` lines so piper stores accuracy/f1 in the run metrics table.

**Papermill params** (override at runtime via pipeline YAML):
- `n_estimators` — number of trees in the Random Forest (default: 100)
- `test_size`    — fraction of data held out for evaluation (default: 0.2)
- `random_state` — reproducibility seed (default: 42)

In [ ]:
# Papermill parameter cell — values below are defaults
n_estimators = 100
test_size    = 0.2
random_state = 42

In [ ]:
import os
import json
import pickle

import numpy as np
from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

output_dir = os.environ.get('PIPER_OUTPUT_DIR', '.')
print(f'output_dir: {output_dir}')

In [ ]:
iris = load_iris()
X, y = iris.data, iris.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=test_size, random_state=random_state, stratify=y
)
print(f'train={len(X_train)}  test={len(X_test)}')

In [ ]:
clf = RandomForestClassifier(n_estimators=n_estimators, random_state=random_state)
clf.fit(X_train, y_train)

In [ ]:
y_pred = clf.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
f1       = f1_score(y_test, y_pred, average='weighted')

# Piper reads these lines and stores them in the run_metrics table
print(f'PIPER_METRIC accuracy={accuracy:.4f}')
print(f'PIPER_METRIC f1={f1:.4f}')
print(f'accuracy={accuracy:.4f}  f1={f1:.4f}')

In [ ]:
os.makedirs(output_dir, exist_ok=True)

# Save model
model_path = os.path.join(output_dir, 'model.pkl')
with open(model_path, 'wb') as f:
    pickle.dump(clf, f)

# Save metrics JSON alongside the model
metrics_path = os.path.join(output_dir, 'metrics.json')
with open(metrics_path, 'w') as f:
    json.dump({'accuracy': accuracy, 'f1': f1,
               'n_estimators': n_estimators, 'test_size': test_size}, f, indent=2)

print(f'model saved  → {model_path}')
print(f'metrics saved → {metrics_path}')